# Le Porte-Parole : API et interface (S0, Seamless, S3)
Lance sur un GPU Colab (T4) l'interface Gradio **et** l'API `/api/translate`, avec un lien public.
Moteurs : `s0` (Whisper small + Opus-MT), `seamless` (SeamlessM4T v2), `s3` (encodeur + projecteur + décodeur). Un seul est en mémoire à la fois.

**Avant de lancer** : Exécution > Modifier le type d'exécution > GPU T4. Mettre `HF_TOKEN` dans les secrets Colab (clé, onglet à gauche), avec accès à l'encodeur Cohere Transcribe Arabic (pour S3). Déposer `speech_app.zip` dans Drive (`MyDrive/speech_app.zip`).

In [ ]:
!nvidia-smi -L
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install fastapi "uvicorn[standard]" python-multipart "gradio>=6.0" "transformers>=5.4.0" accelerate peft sentencepiece protobuf librosa pyyaml sacremoses trafilatura "brotli>=1.2"

In [ ]:
import os, subprocess, sys, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/speech_app.zip'          # <- adapter si besoin
with zipfile.ZipFile(ZIP) as z: z.extractall('/content')
APP = '/content/speech_app' if os.path.isdir('/content/speech_app') else '/content'
if not os.path.isdir('/content/NLP'):
    subprocess.run(['git', 'clone', '-q', '-b', 'benchmarking', 'https://github.com/SamiNakibETU/NLP', '/content/NLP'], check=True)
print('app :', APP, '| dépôt équipe :', os.listdir('/content/NLP')[:6])

In [ ]:
# --- réglages ---
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')            # S3 : encodeur à accès contrôlé
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'   # poids gardés dans Drive : pas de re-téléchargement
os.environ['PORTEPAROLE_NLP_REPO'] = '/content/NLP'
os.environ['PORTEPAROLE_ENGINES'] = 's0,seamless,s3'         # retirer un nom pour le masquer
os.environ['PORTEPAROLE_MT'] = 'off'
os.environ['PORTEPAROLE_API_TOKEN'] = ''                      # mettre un jeton pour protéger l'API (lien public !)
os.environ['PYTHONIOENCODING'] = 'utf-8'

In [ ]:
# Serveur en arrière-plan + tunnel public Cloudflare (sans compte)
import re, time
subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api:app', '--host', '0.0.0.0', '--port', '7860'],
                 cwd=APP, stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tun.stdout:
    if m := re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line):
        URL = m.group(0); break
print('Interface Gradio :', URL)
print('Documentation API :', URL + '/docs')

In [ ]:
# Test depuis Python dans Colab : adresse LOCALE. Le nom public (trycloudflare) ne se résout pas depuis Colab lui-même ;
# il sert à l'extérieur (ton navigateur, une autre machine) : utiliser alors l'URL affichée plus haut.
import requests
LOCAL = 'http://localhost:7860'
H = {'Authorization': 'Bearer ' + os.environ['PORTEPAROLE_API_TOKEN']} if os.environ['PORTEPAROLE_API_TOKEN'] else {}
print(requests.get(LOCAL + '/api/engines', headers=H).json())
# r = requests.post(LOCAL + '/api/translate', headers=H, data={'engine': 'seamless'}, files={'file': open('/content/mon_extrait.mp4', 'rb')})
# print(r.json()['text'])            # ou data={'engine': 's0', 'format': 'srt'} pour recevoir un .srt
# Pour une longue vidéo : data={'wait': 'false'} puis GET /api/jobs/<job_id>

In [ ]:
# Journal du serveur (en cas de souci)
!tail -n 30 /content/server.log